# Diferencias Finitas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurimendiluce/AN2026/blob/main/diferencias_finitas/clase_1.ipynb)


### Antes de empezar (solo si estás en Google Colab)

Colab arranca con un runtime de **Python** por defecto, no de Julia. Para poder correr este notebook ahí es necesario cambiar el entorno de ejecución a Julia.


---

### Setup

Paquetes usados en este notebook.

In [ ]:
using LinearAlgebra, Plots

>Para instalar en Colab algun paquete se debe hacer lo siguiente:
>
>using Pkg
>
>Pkg.add("NombreDelPaquete")

### 1. Discretización de derivadas

Una forma natural de aproximar la derivada de una función $u$ en un punto $x$ es a partir de su definición como límite, reemplazando el límite por un cociente incremental con un paso $h$ finito. Según qué puntos se usen, se obtienen distintas discretizaciones:

- **Diferencias hacia adelante (forward):** $u'(x) \approx \dfrac{u(x+h)-u(x)}{h}$
- **Diferencias hacia atrás (backward):** $u'(x) \approx \dfrac{u(x)-u(x-h)}{h}$
- **Diferencias centradas:** $u'(x) \approx \dfrac{u(x+h)-u(x-h)}{2h}$

Las tres opciones se implementan en una única función, seleccionando el método por un argumento:

In [ ]:
function derivada(u,x;method="forward",h=0.01)
    if method == "forward"
        return (u(x+h)-u(x))/h
    elseif method == "backward"
        return (u(x)-u(x-h))/h
    elseif method == "centradas"
        return (u(x+h)-u(x-h))/(2h)
    end
end

function u(x)
    return sin(x)
end

function ∇u(x)
    return cos(x)
end

Tomando $u(x) = \sin(x)$, cuya derivada exacta es $\cos(x)$, comparamos la aproximación numérica de $u'(1)$ contra el valor exacto para distintos $h$, y estimamos el **orden de convergencia** de cada método (el exponente $p$ tal que el error se comporta como $\mathcal{O}(h^p)$).

En lugar de ajustar una recta global (por ejemplo vía `polyfit`), estimamos el orden **localmente**, comparando cada par de valores consecutivos de $h$. Si el error se comporta como $E(h) \approx C h^p$, entonces

$$ \frac{E(h_i)}{E(h_{i+1})} \approx \left(\frac{h_i}{h_{i+1}}\right)^p \quad \Longrightarrow \quad p \approx \frac{\log\big(E(h_i)/E(h_{i+1})\big)}{\log\big(h_i/h_{i+1}\big)} $$

Esto tiene la ventaja de no depender de un ajuste global: si el orden cambiara para $h$ muy chico (por ejemplo, por efecto del redondeo de punto flotante), lo veríamos reflejado directamente en la tabla de valores locales de $p$, en vez de quedar diluido en una única pendiente promedio.

In [ ]:
function orden(u,∇u,x;method="forward")
    h = [0.1,0.01,0.001,0.0001]
    error = zeros(length(h))
    for i=1:length(h)
        error[i] = abs(∇u(x) - derivada(u,x,h=h[i],method=method))
    end

    println("h\t\terror")
    for i=1:length(h)
        println(h[i], "\t", error[i])
    end
    println()

    println("Orden estimado (pares sucesivos de h):")
    for i=1:length(h)-1
        p = log(error[i]/error[i+1]) / log(h[i]/h[i+1])
        println("  p(h=", h[i], " → h=", h[i+1], ") ≈ ", round(p, digits=3))
    end

    plot(log.(h), log.(error), marker=:circle, xlabel="log(h)", ylabel="log(error)",
         label="método: "*method, legend=:topleft)
end

**Método forward:**

In [ ]:
orden(u,∇u,1)

**Método de diferencias centradas:**

In [ ]:
#metodo centradas
orden(u,∇u,1,method="centradas")

**Conclusión:** las diferencias forward y backward tienen orden 1 (el error decrece linealmente con $h$), mientras que las diferencias centradas alcanzan orden 2 — para el mismo $h$ el error es sensiblemente menor. Esto se debe a que, al combinar $u(x+h)$ y $u(x-h)$ simétricamente, se cancela el término de orden $h$ en el desarrollo de Taylor.

### ¿Qué pasa si $h$ es demasiado chico?

Hasta ahora elegimos $h$ en un rango donde domina el **error de truncamiento** (el que viene de cortar la serie de Taylor), y por eso el error baja con el orden esperado a medida que $h \to 0$. Pero en la práctica $u(x)$ se evalúa con aritmética de punto flotante, de precisión finita: cada evaluación tiene un error de redondeo del orden de la unidad de redondeo de la máquina, $\epsilon_{\text{mach}} \approx 2.22\times10^{-16}$ para `Float64`.

Al calcular $u(x+h)-u(x)$ con $h$ muy chico, restamos dos números muy parecidos: el resultado sufre **cancelación catastrófica**, perdiendo precisión relativa, y ese error se amplifica al dividir por $h$. El error total combina entonces dos efectos que compiten:

$$ E(h) \;\approx\; \underbrace{C_1 h^p}_{\text{truncamiento}} \;+\; \underbrace{C_2\,\frac{\epsilon_{\text{mach}}}{h}}_{\text{redondeo}} $$

Para $h$ grande domina el truncamiento (el error baja al achicar $h$); para $h$ muy chico domina el redondeo (el error empieza a **subir**). Existe entonces un $h$ óptimo, ni muy grande ni muy chico, que minimiza el error total — y a partir de ahí, achicar $h$ deja de ayudar.

In [ ]:
function error_vs_h(u,∇u,x;method="forward")
    h = [10.0^k for k in -1:-1:-16]
    error = [abs(∇u(x) - derivada(u,x,h=hi,method=method)) for hi in h]
    return h, error
end

h_f, err_f = error_vs_h(u,∇u,1,method="forward")
h_c, err_c = error_vs_h(u,∇u,1,method="centradas")

In [ ]:
plot(log10.(h_f), log10.(err_f), marker=:circle, label="forward",
     xlabel="log10(h)", ylabel="log10(error)", legend=:bottomleft)
plot!(log10.(h_c), log10.(err_c), marker=:circle, label="centradas")

In [ ]:
println("h que minimiza el error (numérico):")
println("  forward:   ", h_f[argmin(err_f)])
println("  centradas: ", h_c[argmin(err_c)])
println()
println("Estimación teórica (minimizando C1*h^p + C2*eps/h):")
println("  forward   (p=1) → sqrt(eps)     ≈ ", sqrt(eps(Float64)))
println("  centradas (p=2) → eps^(1/3)     ≈ ", eps(Float64)^(1/3))

**Conclusión:** el gráfico deja de ser una recta para $h$ muy chico: después de bajar con la pendiente esperada (orden 1 o 2, según el método), el error alcanza un mínimo y luego **empieza a crecer**. Ese mínimo aproxima el $h$ óptimo, y coincide razonablemente con la estimación teórica. Notar además que, como la centrada tiene mayor orden, tolera achicar $h$ un poco más antes de que el redondeo tome el control — pero eventualmente el mismo fenómeno aparece en cualquier método.

**Moraleja:** "más chico" no es sinónimo de "más preciso" cuando se trabaja con aritmética de punto flotante; hay un límite práctico impuesto por la precisión de la máquina, más allá del cual seguir refinando $h$ es contraproducente.

### 2. Problemas de valores de contorno: capa límite

Consideremos la ecuación

$$ \varepsilon u_{xx} - u_{x} = f, \qquad u(0)=\alpha, \quad u(1)=\beta. $$

Para el caso $f(x) = 1$, la solución exacta está dada por

$$ u_{\varepsilon}(x) = \alpha + x + (\beta - \alpha -1) \left( \frac{e^{x/\varepsilon}-1 }{e^{1/\varepsilon}-1} \right) $$

> **Nota:** una ecuación de esta forma aparece, por ejemplo, al considerar el estado estacionario ($u_t=0$) de un problema de convección-difusión $u_t = \kappa u_{xx} + a u_x + \phi$, con constantes de difusividad $\kappa>0$ y de convección $a \in \mathbb{R}$. A la proporción $Pe = a/\kappa$ se la conoce como **número de Péclet**, y se toma $\varepsilon = 1/Pe$.

In [ ]:
function u_ε(x,ε;α=1,β=3)
    y=α+x+(β-α-1)*((ℯ^(x/ε)-1)/(ℯ^(1/ε)-1))
    return y
end

Graficamos la solución exacta para $\alpha=1,\ \beta=3$ a medida que $\varepsilon \to 0$:

In [ ]:
x=0:0.01:1

plot(x,u_ε.(x,0.3),label="epsilon=0.3")
plot!(x,u_ε.(x,0.1),label="epsilon=0.1")
plot!(x,u_ε.(x,0.05),label="epsilon=0.05")
plot!(x,u_ε.(x,0.01),label="epsilon=0.01")

A medida que $\varepsilon$ disminuye, la solución desarrolla una transición cada vez más abrupta cerca del borde $x=1$: es la **capa límite**, una región angosta donde la solución cambia muy rápido mientras que en el resto del dominio se mantiene casi constante.

### Resolución numérica

Discretizando con diferencias centradas tanto la derivada primera como la segunda sobre una malla de tamaño $h$, se obtiene un sistema lineal tridiagonal para los valores interiores de $u$:

In [ ]:
function capa_limite(f,N;ε=0.3,α=1,β=3)

    h=1/N
    x=0:h:1
    n=length(x)
    U=zeros(n)
    U[1]=α
    U[n]=β
    A1=(ε/h^2)*Tridiagonal(ones(n-3),-2*ones(n-2),ones(n-3))
    A2=(1/2h)*Tridiagonal(-ones(n-3),zeros(n-2),ones(n-3))
    F=f.(x)[2:n-1]
    F[1]=F[1]-ε*α/h^2+α/2h
    F[end]=F[end]-ε*β/h^2-β/2h

    A=A1-A2
    sol=A\F
    U[2:n-1]=sol
    return U
end

function f(x)
    return -1.0
end

Comparamos la solución numérica con la solución exacta para $\varepsilon = 0.1$ y $N=100$:

In [ ]:
ε=0.1
N=100
x=0:1/N:1
U=capa_limite(f,N,ε=ε)
plot(x,U,label="solucion numerica")
plot!(x,u_ε.(x,ε),label="solucion exacta")

**Conclusión:** la aproximación numérica reproduce bien tanto la zona suave como la capa límite cerca de $x=1$, siempre que la malla sea suficientemente fina en relación a $\varepsilon$. Si $h \gg 2\varepsilon$, el esquema deja de resolver correctamente la capa límite y aparecen oscilaciones espurias — se puede verificar variando `N` y `ε` en la celda anterior.